# CNN TensorFlow

Nguyễn Ngọc Hoàng Nam - B23DCCN585 | Assignment 04

**Bản mở rộng:** notebook này giữ nhóm 18 lượt seed 42 để giải thích thuật toán. Notebook **06** tổng hợp đầy đủ 54 lượt nhiều seed và 18 ablation; notebook **07** thực thi ví dụ số học dùng trong báo cáo 78 trang.

In [1]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Hãy mở notebook từ thư mục repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from src.data import DATASETS, load_data, data_root
print('Python:', sys.version.split()[0])
print('Dữ liệu:', data_root())

Python: 3.10.20
Dữ liệu: E:\PTHTTM\ASG_04_data


## 1. Cách triển khai

Mô hình dùng TensorFlow/Keras Functional API; vòng lặp dùng GradientTape và tf.function. Đầu vào NHWC, chuyển thứ tự kênh trước Flatten để cùng thứ tự đặc trưng với NumPy/PyTorch.

Mã trong các ô dưới lấy trực tiếp từ module trong `src/`. Vòng lặp huấn luyện lưu checkpoint theo validation loss và chỉ đánh giá test sau khi khôi phục checkpoint tốt nhất.

In [2]:
BACKEND = 'tensorflow'
RETRAIN = False  # Đổi thành True để huấn luyện lại 6 cấu hình; sẽ ghi đè kết quả tương ứng.

In [3]:
# Cấu hình GPU trước khi tạo bất kỳ tensor/model nào.
cuda=os.environ.get('CNN_CUDA_DIR','E:/PTHTTM/ASG_04_runtime/cuda/Library/bin')
if os.name=='nt' and Path(cuda).is_dir():
    os.environ['PATH']=cuda+os.pathsep+os.environ['PATH']
    dll_handle=os.add_dll_directory(cuda)
import tensorflow as tf
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu,True)
print('TensorFlow:',tf.__version__)

TensorFlow: 2.10.1


## 2. Mã mô hình

In [4]:
"""TensorFlow/Keras equivalent. Inputs are NHWC; flatten is explicitly NCHW."""
import tensorflow as tf
from tensorflow.keras import layers

def build_model(channels,size,classes,variant='baseline'):
    improved=variant=='improved';x0=tf.keras.Input((size,size,channels));x=x0
    x=layers.Conv2D(8,3,padding='same',name='conv1')(x)
    if improved:x=layers.BatchNormalization(momentum=0.9,epsilon=1e-5,fused=False,name='bn1')(x)
    x=layers.ReLU()(x);x=layers.MaxPool2D()(x)
    x=layers.Conv2D(16,3,padding='same',name='conv2')(x)
    if improved:x=layers.BatchNormalization(momentum=0.9,epsilon=1e-5,fused=False,name='bn2')(x)
    x=layers.ReLU()(x)
    if improved:
        residual=x;x=layers.Conv2D(16,3,padding='same',name='res_conv')(x)
        x=layers.BatchNormalization(momentum=0.9,epsilon=1e-5,fused=False,name='res_bn')(x)
        x=layers.ReLU()(layers.Add()([x,residual]))
    x=layers.MaxPool2D()(x)
    # Match the flattened feature order used by NumPy and PyTorch.
    x=layers.Permute((3,1,2))(x);x=layers.Reshape((16*(size//4)**2,))(x)
    x=layers.Dense(64,name='fc1')(x)
    if improved:x=layers.BatchNormalization(momentum=0.9,epsilon=1e-5,fused=False,name='bn_fc')(x)
    x=layers.ReLU()(x)
    if improved:x=layers.Dropout(0.25)(x)
    out=layers.Dense(classes,name='fc2')(x)
    return tf.keras.Model(x0,out,name='CNN_'+variant)

def load_numpy(model,state,variant):
    if variant=='baseline':mapping=[('conv1',0,'conv'),('conv2',3,'conv'),('fc1',7,'dense'),('fc2',9,'dense')]
    else:mapping=[('conv1',0,'conv'),('bn1',1,'bn'),('conv2',4,'conv'),('bn2',5,'bn'),('res_conv',7,'res_conv'),('res_bn',7,'res_bn'),('fc1',10,'dense'),('bn_fc',11,'bn'),('fc2',14,'dense')]
    for name,i,kind in mapping:
        prefix=f'{i}.'
        if kind=='res_conv':prefix+='conv_';kind='conv'
        if kind=='res_bn':prefix+='bn_';kind='bn'
        if kind=='conv':weights=[state[prefix+'weight'].transpose(2,3,1,0),state[prefix+'bias']]
        elif kind=='dense':weights=[state[prefix+'weight'],state[prefix+'bias']]
        else:weights=[state[prefix+k] for k in ['gamma','beta','running_mean','running_var']]
        model.get_layer(name).set_weights(weights)


## 3. Vòng lặp huấn luyện và đánh giá

Chương trình bên dưới chứa đầy đủ bước lấy batch, forward, loss, gradient, cập nhật, validation, checkpoint và đánh giá test. Các lượt chạy thực tế gọi cùng module trong một tiến trình riêng để cô lập framework.

In [5]:
"""Reproducible full-dataset training. Run: python -m src.train --help."""
import os
for key in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ.setdefault(key,'1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL','2')
import json,csv,time,platform,argparse,sys,copy
from pathlib import Path
import numpy as np
from src.data import ROOT,DATASETS,load_data,batches
from src.numpy_cnn import CNN,cross_entropy,Adam

def classification_metrics(y,logits,k):
    pred=logits.argmax(axis=1);cm=np.bincount(y*k+pred,minlength=k*k).reshape(k,k)
    tp=np.diag(cm).astype(float)
    precision=np.divide(tp,cm.sum(0),out=np.zeros(k),where=cm.sum(0)>0)
    recall=np.divide(tp,cm.sum(1),out=np.zeros(k),where=cm.sum(1)>0)
    f1=np.divide(2*precision*recall,precision+recall,out=np.zeros(k),where=(precision+recall)>0)
    result={'accuracy':float((pred==y).mean()),'macro_precision':float(precision.mean()),'macro_recall':float(recall.mean()),'macro_f1':float(f1.mean()),'test_loss':cross_entropy(logits,y)[0]}
    if k>=5:result['top5_accuracy']=float(np.any(np.argsort(logits,axis=1)[:,-5:]==y[:,None],axis=1).mean())
    return result,cm

def experiment_dir(backend,dataset,variant,seed=42,ablation=None):
    name=f'{dataset}_{backend}_{variant}'
    if ablation:return ROOT/'results'/'ablation'/f'seed_{seed}'/(name+'_'+ablation)
    if seed!=42:return ROOT/'results'/'multiseed'/f'seed_{seed}'/name
    return ROOT/'results'/name

def _train(backend,dataset,variant,epochs=None,batch_size=128,seed=42,force=False,ablation=None):
    if ablation and (backend!='pytorch' or variant!='improved'):
        raise ValueError('Ablation requires the PyTorch improved architecture.')
    cfg=DATASETS[dataset];epochs=epochs or cfg['epochs']
    out=experiment_dir(backend,dataset,variant,seed,ablation);out.mkdir(parents=True,exist_ok=True)
    if (out/'metrics.json').exists() and not force:
        saved=json.loads((out/'config.json').read_text())
        for key,value in dict(seed=seed,epochs=epochs,batch_size=batch_size,ablation=ablation).items():
            if saved.get(key)!=value:raise ValueError(f'Existing {out}: {key} differs; use --force or another seed.')
        return json.loads((out/'metrics.json').read_text())
    data=load_data(dataset);x,y=data['x'],data['y'];ti,vi=data['train_ids'],data['val_ids']
    spec=dict(channels=cfg['channels'],size=cfg['size'],classes=cfg['classes'],variant=variant,seed=seed)
    initial=CNN(**spec);state=initial.state();params=initial.parameter_count()
    np.random.seed(seed)
    if backend=='numpy':
        model=initial;optimizer=Adam();device='CPU';version=np.__version__
        def predict(b):return model.forward(b,False)
        def step(b,t):
            logits=model.forward(b,True);loss,g=cross_entropy(logits,t);model.backward(g);optimizer.step(model.params())
            return loss,int((logits.argmax(1)==t).sum())
        def save():np.savez_compressed(out/'weights.npz',**model.state())
        def restore():
            with np.load(out/'weights.npz') as d:model.load_state(dict(d))
    elif backend=='pytorch':
        import torch
        from src.torch_cnn import TorchCNN
        torch.manual_seed(seed);torch.set_num_threads(4)
        torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
        torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
        device='cuda' if torch.cuda.is_available() else 'cpu';version=torch.__version__
        model=TorchCNN(cfg['channels'],cfg['size'],cfg['classes'],variant);model.load_numpy(state);model.to(device)
        assert sum(p.numel() for p in model.parameters())==params
        model.ablate(ablation);params=sum(p.numel() for p in model.parameters())
        optimizer=torch.optim.Adam(model.parameters(),lr=1e-3,eps=1e-8)
        def predict(b):
            model.eval()
            with torch.no_grad():return model(torch.from_numpy(np.ascontiguousarray(b)).to(device)).cpu().numpy()
        def step(b,t):
            model.train();optimizer.zero_grad(set_to_none=True)
            logits=model(torch.from_numpy(np.ascontiguousarray(b)).to(device));target=torch.from_numpy(t).to(device)
            loss=torch.nn.functional.cross_entropy(logits,target);loss.backward();optimizer.step()
            return loss.item(),int((logits.argmax(1)==target).sum().item())
        def save():torch.save(model.state_dict(),out/'weights.pt')
        def restore():model.load_state_dict(torch.load(out/'weights.pt',map_location=device,weights_only=True))
    elif backend=='tensorflow':
        cuda=os.environ.get('CNN_CUDA_DIR','E:/PTHTTM/ASG_04_runtime/cuda/Library/bin')
        if os.name=='nt' and Path(cuda).is_dir():
            os.environ['PATH']=cuda+os.pathsep+os.environ['PATH'];dll=os.add_dll_directory(cuda)
        import tensorflow as tf
        from src.tf_cnn import build_model,load_numpy
        tf.keras.utils.set_random_seed(seed)
        tf.config.threading.set_intra_op_parallelism_threads(4);tf.config.threading.set_inter_op_parallelism_threads(2)
        tf.config.experimental.enable_tensor_float_32_execution(False)
        gpus=tf.config.list_physical_devices('GPU')
        for gpu in gpus:tf.config.experimental.set_memory_growth(gpu,True)
        device='GPU' if gpus else 'CPU';version=tf.__version__
        model=build_model(cfg['channels'],cfg['size'],cfg['classes'],variant);load_numpy(model,state,variant)
        assert sum(int(np.prod(v.shape)) for v in model.trainable_weights)==params
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3,epsilon=1e-8)
        @tf.function(reduce_retracing=True)
        def train_step(b,t):
            with tf.GradientTape() as tape:
                logits=model(b,training=True)
                loss=tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=t,logits=logits))
            optimizer.apply_gradients(zip(tape.gradient(loss,model.trainable_weights),model.trainable_weights))
            return loss,tf.reduce_sum(tf.cast(tf.argmax(logits,axis=1)==t,tf.int32))
        @tf.function(reduce_retracing=True)
        def infer(b):return model(b,training=False)
        def predict(b):return infer(np.ascontiguousarray(b.transpose(0,2,3,1))).numpy()
        def step(b,t):
            loss,correct=train_step(np.ascontiguousarray(b.transpose(0,2,3,1)),t)
            return float(loss.numpy()),int(correct.numpy())
        def save():model.save_weights(str(out/'weights.h5'))
        def restore():model.load_weights(str(out/'weights.h5'))
    else:raise ValueError(backend)
    history=[];best=float('inf');best_epoch=0;train_seconds=0;val_seconds=0
    print(f'START {dataset} {backend} {variant} seed={seed} ablation={ablation}: train={len(ti)} val={len(vi)} epochs={epochs} device={device} params={params}',flush=True)
    config=dict(dataset=dataset,backend=backend,variant=variant,epochs=epochs,batch_size=batch_size,seed=seed,learning_rate=0.001,optimizer='Adam',adam_beta1=0.9,adam_beta2=0.999,adam_epsilon=1e-8,normalization='uint8 / 255',train_samples=len(ti),validation_samples=len(vi),test_samples=len(data['y_test']),device=device,framework_version=version,python=sys.version,platform=platform.platform(),parameter_count=params,selection='minimum validation cross-entropy',augmentation=False,training_scope='all predefined training samples, no subsampling')
    from threadpoolctl import threadpool_info
    config['ablation']=ablation
    config['split_seed']=42
    config['cpu_thread_pools']=threadpool_info()
    (out/'config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    for epoch in range(1,epochs+1):
        ts=time.perf_counter();ls=0;correct=0
        for b,t in batches(x,y,ti,batch_size,seed+epoch):
            loss,c=step(b,t);ls+=loss*len(t);correct+=c
        elapsed=time.perf_counter()-ts;train_seconds+=elapsed;ts=time.perf_counter()
        logits=np.concatenate([predict(b) for b,t in batches(x,y,vi,batch_size)])
        vl=cross_entropy(logits,y[vi])[0];va=float((logits.argmax(1)==y[vi]).mean());ve=time.perf_counter()-ts;val_seconds+=ve
        if not np.isfinite(ls+vl):raise FloatingPointError('Non-finite loss. Stop rather than record invalid results.')
        if vl<best:best=vl;best_epoch=epoch;save()
        row=dict(epoch=epoch,train_loss=ls/len(ti),train_accuracy=correct/len(ti),val_loss=vl,val_accuracy=va,train_seconds=elapsed,validation_seconds=ve)
        history.append(row)
        with (out/'history.csv').open('w',newline='',encoding='utf-8') as f:
            writer=csv.DictWriter(f,fieldnames=list(row));writer.writeheader();writer.writerows(history)
        print(f'{dataset}/{backend}/{variant} epoch {epoch}/{epochs}: loss {row["train_loss"]:.4f} val_loss {vl:.4f} val_acc {va:.4f} train_s {elapsed:.1f}',flush=True)
    restore();xt,yt=data['x_test'],data['y_test'];ts=time.perf_counter()
    logits=np.concatenate([predict(b) for b,t in batches(xt,yt,np.arange(len(yt)),batch_size)])
    infer_seconds=time.perf_counter()-ts
    metrics,cm=classification_metrics(yt,logits,cfg['classes'])
    metrics.update({k:config[k] for k in ['dataset','backend','variant','parameter_count','train_samples','validation_samples','test_samples','device']})
    metrics.update(seed=seed,ablation=ablation,epochs=epochs,best_epoch=best_epoch,best_val_loss=best,train_seconds=train_seconds,validation_seconds=val_seconds,test_seconds=infer_seconds)
    shifted=logits-logits.max(1,keepdims=True);probs=np.exp(shifted);probs/=probs.sum(1,keepdims=True)
    with (out/'predictions.csv').open('w',newline='',encoding='utf-8') as f:
        writer=csv.writer(f);writer.writerow(['test_id','true_label','predicted_label','confidence'])
        writer.writerows(zip(range(len(yt)),yt.tolist(),logits.argmax(1).tolist(),probs.max(1).tolist()))
    np.savez_compressed(out/'test_outputs.npz',logits=logits,labels=yt)
    np.savetxt(out/'confusion_matrix.csv',cm,delimiter=',',fmt='%d')
    (out/'metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    print('COMPLETE '+json.dumps(metrics),flush=True)
    return metrics

def train(backend,dataset,variant,epochs=None,batch_size=128,seed=42,force=False,ablation=None):
    # A notebook and the CLI may request the same configuration concurrently.
    # Serialize that configuration; after waiting, reuse its complete results.
    from filelock import FileLock
    folder=experiment_dir(backend,dataset,variant,seed,ablation)
    folder.mkdir(parents=True,exist_ok=True)
    with FileLock(str(folder/'.train.lock'),timeout=7200):
        return _train(backend,dataset,variant,epochs,batch_size,seed,force,ablation)



In [6]:
def execute_dataset(name):
    rows=[]
    for variant in ['baseline','improved']:
        folder=ROOT/'results'/f'{name}_{BACKEND}_{variant}'
        if RETRAIN or not (folder/'metrics.json').exists():
            cmd=[sys.executable,'-m','src.train','--backend',BACKEND,'--dataset',name,'--variant',variant]
            if RETRAIN:cmd.append('--force')
            subprocess.run(cmd,cwd=ROOT,check=True)
        else:
            print('Đọc kết quả đã huấn luyện:',folder.name)
        rows.append(json.loads((folder/'metrics.json').read_text()))
        display(pd.read_csv(folder/'history.csv'))
    display(pd.DataFrame(rows)[['dataset','variant','accuracy','macro_f1','test_loss','top5_accuracy','best_epoch','train_seconds']])
    return rows

## 4. MNIST

In [7]:
mnist_results=execute_dataset('mnist')

Đọc kết quả đã huấn luyện: mnist_tensorflow_baseline


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,0.266692,0.922724,0.105824,0.965328,10.137760,0.416640
1,2,0.079407,0.975537,0.076774,0.976496,1.670353,0.107570
2,3,0.056563,0.983167,0.070482,0.978163,1.680218,0.101256
3,4,0.045179,0.986593,0.062512,0.980163,1.658771,0.091643
4,5,0.038664,0.987889,0.052450,0.983164,1.698345,0.096050


Đọc kết quả đã huấn luyện: mnist_tensorflow_improved


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,0.259053,0.931353,0.079914,0.977996,9.572932,0.435283
1,2,0.078917,0.978519,0.054309,0.984831,3.101229,0.124918
2,3,0.056074,0.983648,0.051579,0.984164,3.146989,0.128224
3,4,0.044095,0.986945,0.045940,0.985664,3.141661,0.127199
4,5,0.036458,0.988630,0.046114,0.986664,3.155821,0.127486


,dataset,variant,accuracy,macro_f1,test_loss,top5_accuracy,best_epoch,train_seconds
0,mnist,baseline,0.9868,0.986695,0.040432,0.9997,5,16.845447
1,mnist,improved,0.9875,0.987486,0.037104,1.0000,4,22.118631


## 5. CIFAR-10

In [8]:
cifar10_results=execute_dataset('cifar10')

Đọc kết quả đã huấn luyện: cifar10_tensorflow_baseline


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,1.662094,0.404489,1.464885,0.4706,6.844864,0.470852
1,2,1.380528,0.510644,1.337228,0.5298,2.429974,0.197867
2,3,1.273136,0.549867,1.271018,0.5488,2.697191,0.215578
3,4,1.200964,0.579667,1.198066,0.5758,2.909260,0.222417
4,5,1.151613,0.594644,1.164644,0.5890,2.814790,0.201397
5,6,1.097881,0.614133,1.141779,0.6016,2.902168,0.230975
6,7,1.060548,0.628778,1.110404,0.6144,3.065402,0.272185
7,8,1.024473,0.641244,1.089639,0.6164,2.981157,0.225784
8,9,0.996308,0.652311,1.074855,0.6238,2.961393,0.221312
9,10,0.966062,0.663111,1.065342,0.6288,3.503165,0.264746


Đọc kết quả đã huấn luyện: cifar10_tensorflow_improved


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,1.585895,0.434778,1.354051,0.5200,11.234363,0.583928
1,2,1.253127,0.553622,1.153248,0.5896,3.953354,0.220877
2,3,1.127990,0.598444,1.085273,0.6134,3.584032,0.217078
3,4,1.050065,0.628333,1.105364,0.6020,3.614989,0.229933
4,5,0.994236,0.645533,1.022968,0.6352,3.590015,0.226842
5,6,0.953779,0.663444,1.006872,0.6484,3.619205,0.214978
6,7,0.909273,0.677822,1.063600,0.6236,3.625521,0.218785
7,8,0.882866,0.688222,0.960727,0.6628,3.616354,0.223358
8,9,0.852330,0.699556,0.966590,0.6660,3.702415,0.212996
9,10,0.829025,0.707400,1.068366,0.6312,3.675654,0.216214


,dataset,variant,accuracy,macro_f1,test_loss,top5_accuracy,best_epoch,train_seconds
0,cifar10,baseline,0.6301,0.626440,1.053614,0.9636,10,33.109362
1,cifar10,improved,0.6602,0.658797,0.963178,0.9683,8,44.215902


## 6. CIFAR-100

In [9]:
cifar100_results=execute_dataset('cifar100')

Đọc kết quả đã huấn luyện: cifar100_tensorflow_baseline


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,4.190032,0.069622,3.770262,0.1276,10.134980,0.464457
1,2,3.577844,0.162089,3.443115,0.1708,2.848383,0.251428
2,3,3.329286,0.204644,3.304510,0.2022,3.049349,0.263193
3,4,3.183910,0.232667,3.205292,0.2200,3.088037,0.239964
4,5,3.072545,0.253311,3.153987,0.2336,3.021113,0.220279
5,6,2.998023,0.267378,3.096366,0.2510,2.820955,0.215237
6,7,2.923357,0.284400,3.075997,0.2482,3.084556,0.258427
7,8,2.867416,0.291911,3.069618,0.2430,2.874869,0.211459
8,9,2.816706,0.305556,2.992172,0.2592,2.901745,0.237701
9,10,2.771337,0.312600,2.960217,0.2728,2.811515,0.212289


Đọc kết quả đã huấn luyện: cifar100_tensorflow_improved


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,train_seconds,validation_seconds
0,1,4.131924,0.088844,3.631814,0.1602,10.069190,0.555913
1,2,3.550690,0.166956,3.301076,0.2204,3.850471,0.253077
2,3,3.273045,0.211956,3.096254,0.2490,3.799785,0.238959
3,4,3.094131,0.244133,3.015181,0.2614,3.792977,0.252839
4,5,2.969853,0.266378,2.903692,0.2800,3.733550,0.250527
5,6,2.870628,0.285467,2.836092,0.3018,3.745798,0.243188
6,7,2.789425,0.302822,2.798424,0.3046,3.819933,0.249343
7,8,2.724816,0.312733,2.840616,0.2956,3.825447,0.239389
8,9,2.675141,0.322133,2.760688,0.3088,3.766771,0.241343
9,10,2.617875,0.336022,2.804024,0.3022,3.881230,0.244380


,dataset,variant,accuracy,macro_f1,test_loss,top5_accuracy,best_epoch,train_seconds
0,cifar100,baseline,0.2854,0.274375,2.907140,0.5819,11,42.634300
1,cifar100,improved,0.3254,0.312731,2.690552,0.6285,11,51.993648


## 7. Kiểm tra triển khai

In [10]:
verification=ROOT/'results'/f'verification_{BACKEND}.json'
if verification.exists():
    display(pd.DataFrame(json.loads(verification.read_text())))
else:
    print(subprocess.run([sys.executable,'tools/run_test_suite.py'],cwd=ROOT,capture_output=True,text=True,check=True).stderr)

,backend,dataset,variant,passed,max_absolute_errors
0,tensorflow,mnist,baseline,True,"{'train_logits': 1.5497207641601562e-06, 'inpu..."
1,tensorflow,mnist,improved,True,"{'train_logits': 3.337860107421875e-06, 'input..."
2,tensorflow,cifar10,baseline,True,"{'train_logits': 2.4437904357910156e-06, 'inpu..."
3,tensorflow,cifar10,improved,True,"{'train_logits': 3.6954879760742188e-06, 'inpu..."
4,tensorflow,cifar100,baseline,True,"{'train_logits': 3.0994415283203125e-06, 'inpu..."
5,tensorflow,cifar100,improved,True,"{'train_logits': 5.0067901611328125e-06, 'inpu..."


## Diễn giải

So sánh baseline với improved trong cùng dataset/backend. Accuracy không phản ánh toàn bộ chất lượng ở CIFAR-100: cần xem macro-F1, top-5 và các lớp hay nhầm. Train metrics được tích lũy trong lúc cập nhật trọng số, có dropout ở improved; validation chạy ở chế độ eval, nên không thể diễn giải mọi chênh lệch train-validation là overfitting.

Các chỉ số là một lượt chạy seed 42. Thời gian gồm bước huấn luyện thực tế nhưng không phải benchmark phần cứng độc lập. Notebook 05 đưa ra so sánh chung dựa trên 18 kết quả.